# Week 3 — Chapter 2 Practice

Inspecting initial weights in Keras and a manually implemented model.

Restored from the completed classroom exercises and rerun after the unsaved Colab edits were lost. The outputs below are from this rerun; random initial weights may differ from the classroom screenshots.

Reference: [Chapter 2 companion notebook](https://github.com/fchollet/deep-learning-with-python-notebooks/blob/master/chapter02_mathematical-building-blocks.ipynb). The NaiveDense and NaiveSequential implementations follow that notebook.

## 1. Inspect the Keras model

In [1]:
import keras
from keras import layers, ops

model = keras.Sequential([
    keras.Input(shape=(784,)),
    layers.Dense(512, activation="relu"),
    layers.Dense(10, activation="softmax")
])
W, b = model.layers[0].get_weights()
print("Weight shape:", W.shape)
print("Bias shape:", b.shape)
print("Sample weights:")
print(W[:5, :3])
assert W.shape == (784, 512) and b.shape == (512,)


Weight shape: (784, 512)
Bias shape: (512,)
Sample weights:
[[ 0.04282896 -0.06268548  0.02609433]
 [-0.02051374  0.03006107  0.06606065]
 [ 0.00856453 -0.0073512   0.02830547]
 [-0.0162039   0.00241718  0.05663341]
 [-0.0051965  -0.03507777 -0.04023834]]


## 2. Define the manually implemented layers and model

In [2]:
class NaiveDense:
    def __init__(self, input_size, output_size, activation=None):
        self.activation = activation
        self.W = keras.Variable(shape=(input_size, output_size), initializer="uniform")
        self.b = keras.Variable(shape=(output_size,), initializer="zeros")

    def __call__(self, inputs):
        x = ops.matmul(inputs, self.W) + self.b
        return self.activation(x) if self.activation is not None else x

    @property
    def weights(self):
        return [self.W, self.b]

class NaiveSequential:
    def __init__(self, layers):
        self.layers = layers

    def __call__(self, inputs):
        x = inputs
        for layer in self.layers:
            x = layer(x)
        return x

    @property
    def weights(self):
        return [weight for layer in self.layers for weight in layer.weights]

naive_model = NaiveSequential([
    NaiveDense(784, 512, activation=ops.relu),
    NaiveDense(512, 10, activation=ops.softmax)
])
assert len(naive_model.weights) == 4


In [3]:
first_layer = naive_model.layers[0]
print("Weight shape:", first_layer.W.shape)
print("Bias shape:", first_layer.b.shape)
print("Sample weights:")
print(first_layer.W.numpy()[:5, :3])
assert tuple(first_layer.W.shape) == (784, 512)
assert tuple(first_layer.b.shape) == (512,)


Weight shape: (784, 512)
Bias shape: (512,)
Sample weights:
[[-0.0444121   0.030887    0.00980176]
 [ 0.00828637 -0.01882801  0.01619449]
 [-0.0189853   0.01502532 -0.02603394]
 [ 0.02589056 -0.04786311  0.03342749]
 [-0.0346234   0.0227023  -0.04186153]]


## Conclusion
Both models have a first-layer weight shape of (784, 512) and a bias shape of (512,). Their shapes match because both layers connect 784 inputs to 512 neurons. Random initialization can produce different values. These are initial parameters; this notebook does not train either model.